# CNeuroMod QA — tSNR brain maps

Volumetric montages of average temporal-SNR maps (MNI space): one panel per subject, one per dataset, and one grand average pooling every subject across every dataset — read directly from the `tsnr` derivative of `source_data/cneuromod.all/{dataset}/`. Figures are written to `output_data/figures/tsnr_maps/`.

tSNR is volumetric and QA cares about signal dropout in ventral/orbitofrontal, temporal and subcortical regions, so we render faithful volumetric slices (nilearn) rather than a cortical surface, which would discard subcortex/cerebellum.

In [1]:
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import nibabel as nib
import numpy as np
from nilearn import plotting

# The shared house style lives next to the notebooks (nbconvert runs with
# notebooks/ as the cwd, so `analysis` is not importable here); the sys.path
# line makes the import work when the notebook is opened from the repo root too.
sys.path.insert(0, str(Path.cwd()))
import figure_style  # noqa: E402

# Paths are provided by `invoke run-notebooks` as environment variables.
# Figures go in output_data/figures/{FIG_NAME}/ (also the notebook's "already
# ran" sentinel); the avgtsnr maps this notebook reads are never persisted —
# they live only in the source `tsnr` derivative, fetched by `invoke fetch`.
FIG_NAME = "tsnr_maps"
OUTPUT_DIR = Path(os.environ.get("OUTPUT_DATA_DIR", "../output_data"))
SOURCE_DIR = Path(os.environ.get("SOURCE_DATA_DIR", "../source_data")) / "cneuromod.all"
FIG_DIR = OUTPUT_DIR / "figures" / FIG_NAME
FIG_DIR.mkdir(parents=True, exist_ok=True)

# The upstream per-subject average tSNR map, MNI space (see analysis/tsnr_maps.py).
SPACE = "MNI152NLin2009cAsym"
SUBJECT_AVG_GLOB = f"sub-*/sub-*_space-{SPACE}_stat-avgtsnr_statmap.nii.gz"

# Shared display settings so panels are visually comparable.
CMAP = "inferno"

# nilearn's own defaults for the slice annotations (L/R, z=…) and the panel
# title, used for every montage the composed figure does not place.
DEFAULT_ANNOTATION_SIZE = 12
DEFAULT_TITLE_SIZE = 15

# Fixed cut coordinates, used unchanged for every panel this notebook renders
# (every dataset, every subject, the grand average, and every coverage panel)
# and for all three views below. Hard-coded rather than picked per image
# (`nilearn.plotting.find_cut_slices`) so every panel across every dataset is
# sliced identically and directly comparable. Axial (CUT_COORDS) was chosen
# once by inspecting the grand-average tSNR map: `find_cut_slices(
# grand_average, direction="z", n_cuts=8)` gave (-28.5, -14.5, -0.5, 19.5,
# 33.5, 47.5, 59.5, 71.5), which we extended with two fixed inferior slices
# (-54, -42) so the cerebellum (vermis/tonsils) — which `find_cut_slices`
# alone never reaches — is always shown too. The 19.5 slice was later moved
# to 15 by hand. Sagittal (SAGITTAL_CUT_COORDS,
# direction "x") and coronal (CORONAL_CUT_COORDS, direction "y") were each
# chosen the same way, straight from `find_cut_slices(grand_average,
# direction=<"x"|"y">, n_cuts=8)`, with no manual extension needed.
CUT_COORDS = (-54, -42, -28.5, -14.5, -0.5, 15, 33.5, 47.5, 59.5, 71.5)
SAGITTAL_CUT_COORDS = (-64.5, -48.5, -34.5, -14.5, 5.5, 21.5, 37.5, 59.5)
CORONAL_CUT_COORDS = (-90.5, -74.5, -52.5, -28.5, -14.5, -0.5, 15.5, 51.5)

# Every panel is rendered once per view: axial keeps the historical unsuffixed
# filename (`{name}.png`), sagittal/coronal add a `_sagittal`/`_coronal`
# filename suffix so all three sit side by side in FIG_DIR.
VIEWS = (
    ("", "z", CUT_COORDS),
    ("_sagittal", "x", SAGITTAL_CUT_COORDS),
    ("_coronal", "y", CORONAL_CUT_COORDS),
)

# Fixed color ranges (rather than a per-run computed ceiling) so panels stay
# numerically comparable across notebook re-runs and across datasets.
TSNR_VMIN, TSNR_VMAX = 0, 50
COVERAGE_VMIN, COVERAGE_VMAX = 0, 1
# Coverage overlay transparency, so the anatomical background (sulci/ventricles)
# stays visible under the yellow "fully covered" regions instead of being fully
# occluded.
COVERAGE_ALPHA = 0.5


def montage_figure(out_path):
    """`(figure, annotation_size, title_size)` for the montage written to `out_path`.

    A panel the composed figure places gets an explicit figure at exactly the
    box size it is placed in, so it lands on the page 1:1 — roughly 4.1 inches
    wide, a 5.5x linear reduction from nilearn's default, which the slice
    annotations and title have to shrink with. Every other panel (per subject,
    sagittal, coronal) passes `figure=None` and keeps nilearn's own sizing.
    """
    size = figure_style.panel_size(f"{FIG_NAME}/{out_path.name}", None)
    if size is None:
        return None, DEFAULT_ANNOTATION_SIZE, DEFAULT_TITLE_SIZE
    annotation_size, title_size = figure_style.montage_font_sizes(size[0])
    return plt.figure(figsize=size, facecolor="k"), annotation_size, title_size


def montage_title(title, short_title, placed):
    """The title to draw: the short form on a placed panel, the full one otherwise.

    nilearn draws the title in an opaque box, and on a placed panel that box is
    wide enough to cover the first slice or two — including anything the
    composed figure annotates there by hand. A placed panel also sits under a
    caption that already supplies the context the long title repeats, so the
    short form loses nothing there.
    """
    return short_title if (placed and short_title is not None) else title


def size_colorbar(display, labelsize):
    """Match the colorbar tick labels to the rest of the panel's annotations."""
    colorbar = getattr(display, "_cbar", None)
    if colorbar is not None:
        colorbar.ax.tick_params(labelsize=labelsize)

In [2]:
from nilearn.image import resample_to_img


def discover_datasets():
    """Dataset names under SOURCE_DIR whose tsnr derivative has >=1 avgtsnr map."""
    datasets = []
    if SOURCE_DIR.is_dir():
        for tsnr_dir in sorted(SOURCE_DIR.glob("*/tsnr")):
            maps = [p for p in tsnr_dir.glob(SUBJECT_AVG_GLOB) if p.is_file()]
            if maps:
                datasets.append(tsnr_dir.parent.name)
    return datasets


def subject_maps(dataset):
    """Sorted per-subject avgtsnr map paths for one dataset, read from source_data."""
    tsnr_dir = SOURCE_DIR / dataset / "tsnr"
    return sorted(p for p in tsnr_dir.glob(SUBJECT_AVG_GLOB) if p.is_file())


def average_image(paths):
    """Mean tSNR image over a list of subject maps, computed in memory (nothing written).

    Maps may sit on slightly different grids, so each is resampled to the first
    map's grid before averaging. NaNs are ignored voxelwise so a subject missing
    coverage never blanks a voxel for everyone.
    """
    reference = nib.load(str(paths[0]))
    stack = []
    for path in paths:
        image = nib.load(str(path))
        if image.shape != reference.shape or not np.allclose(image.affine, reference.affine):
            image = resample_to_img(image, reference, copy_header=True)
        stack.append(np.asarray(image.dataobj, dtype=np.float32))
    mean = np.nanmean(np.stack(stack, axis=-1), axis=-1)
    return nib.Nifti1Image(mean, reference.affine, reference.header)


def dataset_average_image(dataset):
    """Mean tSNR image over one dataset's subject maps."""
    return average_image(subject_maps(dataset))


datasets = discover_datasets()
print(f"found avgtsnr maps for: {datasets}")
if not datasets:
    print("No tSNR maps found — run `invoke fetch` first (needs data access).")
dataset_images = {dataset: dataset_average_image(dataset) for dataset in datasets}

# Grand average: every subject from every (light-v1) dataset pooled into one
# mean map, each subject weighted equally regardless of how many subjects its
# dataset contributes — a single cross-dataset "house average" tSNR panel.
all_subject_paths = [path for dataset in datasets for path in subject_maps(dataset)]
grand_average_image = average_image(all_subject_paths) if all_subject_paths else None


found avgtsnr maps for: ['floc', 'retinotopy', 'things']


In [3]:
from nilearn.datasets import fetch_icbm152_2009

# Two coverage thresholds: 30 (the original, stricter "good signal" cut) and
# 10 (a looser "any usable signal" cut, useful for spotting regions with
# near-total dropout that the 30 threshold already shows as uncovered
# everywhere). Filenames keep no suffix for THRESHOLD 30 (unchanged from
# before) and get a "_thr{threshold}" suffix for every other threshold.
THRESHOLDS = (30, 10)
COVERAGE_CMAP = "viridis"
NILEARN_DIR = Path(os.environ.get("SOURCE_DATA_DIR", "../source_data")) / "nilearn"


def coverage_suffix(threshold):
    return "" if threshold == 30 else f"_thr{threshold}"


def binary_coverage_image(paths, threshold, brain_mask_path):
    """Fraction of subjects with tsnr above `threshold` at each voxel, in-brain only.

    Same load/resample loop as `average_image`, but each subject's map is
    thresholded to a 0/1 mask before averaging — turning "how good is signal"
    into "how many subjects had usable signal at all", a direct coverage/
    dropout QA signal. The averaged fraction is then zeroed outside the MNI
    whole-brain mask: a low tSNR threshold alone is not enough to exclude
    background/skull voxels, whose thermal-noise tSNR routinely clears it too.
    """
    reference = nib.load(str(paths[0]))
    stack = []
    for path in paths:
        image = nib.load(str(path))
        if image.shape != reference.shape or not np.allclose(image.affine, reference.affine):
            image = resample_to_img(image, reference, copy_header=True)
        data = np.asarray(image.dataobj, dtype=np.float32)
        stack.append((data > threshold).astype(np.float32))
    mean = np.nanmean(np.stack(stack, axis=-1), axis=-1)

    mask_image = resample_to_img(
        nib.load(str(brain_mask_path)), reference,
        interpolation="nearest", copy_header=True,
    )
    in_brain = np.asarray(mask_image.dataobj) > 0
    mean = np.where(in_brain, mean, 0.0)
    return nib.Nifti1Image(mean, reference.affine, reference.header)


# The MNI152 T1 template (anatomical background) and whole-brain mask (to
# exclude background/skull voxels from coverage, see binary_coverage_image)
# are used below. `invoke fetch` caches them under NILEARN_DIR (see
# analysis/mni152.py, whose fetch_mni152_templates wraps the same call);
# nilearn checks the cache before downloading, so this is a cheap no-op when
# already fetched. If unavailable, warn and skip the coverage panels rather
# than raising, matching this notebook's tolerant style.
try:
    mni_templates = fetch_icbm152_2009(data_dir=str(NILEARN_DIR))
    mni_t1_path = Path(mni_templates["t1"])
    mni_mask_path = Path(mni_templates["mask"])
except Exception as error:
    print(f"⚠️  MNI152 template unavailable ({error}) — run `invoke fetch` first. "
          "Skipping tSNR coverage panels.")
    mni_t1_path = None
    mni_mask_path = None

if mni_t1_path is not None:
    dataset_coverage_images = {
        threshold: {
            dataset: binary_coverage_image(subject_maps(dataset), threshold, mni_mask_path)
            for dataset in datasets
        }
        for threshold in THRESHOLDS
    }
    grand_average_coverage_images = {
        threshold: (
            binary_coverage_image(all_subject_paths, threshold, mni_mask_path)
            if all_subject_paths else None
        )
        for threshold in THRESHOLDS
    }


[fetch_icbm152_2009] Dataset directory found: /home/pbellec/git/cneuromod.all.qa_figures/source_data/nilearn/icbm152_2009


In [4]:
def plot_coverage(stat_map, title, out_path, bg_img, display_mode, cut_coords,
                  short_title=None):
    """Montage of a 0-1 subject-coverage map on an MNI152 anatomical background."""
    figure, annotation_size, title_size = montage_figure(out_path)
    title = montage_title(title, short_title, figure is not None)
    display = plotting.plot_stat_map(
        stat_map, bg_img=bg_img, display_mode=display_mode, cut_coords=cut_coords,
        cmap=COVERAGE_CMAP, vmin=COVERAGE_VMIN, vmax=COVERAGE_VMAX, threshold=0.01,
        colorbar=True, black_bg=True, symmetric_cbar=False,
        transparency=COVERAGE_ALPHA, figure=figure, annotate=False, title=None,
    )
    display.annotate(size=annotation_size)
    display.title(title, size=title_size)
    size_colorbar(display, annotation_size)
    display.savefig(str(out_path), dpi=figure_style.PAGE_DPI)
    display.close()


# One coverage montage per dataset, per threshold, per view (fraction of
# subjects with tsnr > threshold).
if mni_t1_path is not None:
    for threshold in THRESHOLDS:
        threshold_suffix = coverage_suffix(threshold)
        for dataset, image in dataset_coverage_images[threshold].items():
            for view_suffix, display_mode, cut_coords in VIEWS:
                plot_coverage(
                    image, f"{dataset} — tSNR coverage (>{threshold})",
                    FIG_DIR / f"{dataset}_coverage{threshold_suffix}{view_suffix}.png",
                    bg_img=str(mni_t1_path), display_mode=display_mode, cut_coords=cut_coords,
                )

        # Grand-average coverage montage, pooling every subject across every dataset.
        grand_average_coverage_image = grand_average_coverage_images[threshold]
        if grand_average_coverage_image is not None:
            for view_suffix, display_mode, cut_coords in VIEWS:
                plot_coverage(
                    grand_average_coverage_image, f"all datasets — tSNR coverage (>{threshold})",
                    FIG_DIR / f"all_datasets_coverage{threshold_suffix}{view_suffix}.png",
                    bg_img=str(mni_t1_path), display_mode=display_mode, cut_coords=cut_coords,
                    short_title=f"coverage >{threshold}",
                )


/home/pbellec/git/cneuromod.all.qa_figures/.venv/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:2599: RuntimeWarning: invalid value encountered in <lambda> (vectorized)
  outputs = ufunc(*args, out=...)


/home/pbellec/git/cneuromod.all.qa_figures/.venv/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:2599: RuntimeWarning: invalid value encountered in <lambda> (vectorized)
  outputs = ufunc(*args, out=...)


/home/pbellec/git/cneuromod.all.qa_figures/.venv/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:2599: RuntimeWarning: invalid value encountered in <lambda> (vectorized)
  outputs = ufunc(*args, out=...)


/home/pbellec/git/cneuromod.all.qa_figures/.venv/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:2599: RuntimeWarning: invalid value encountered in <lambda> (vectorized)
  outputs = ufunc(*args, out=...)


/home/pbellec/git/cneuromod.all.qa_figures/.venv/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:2599: RuntimeWarning: invalid value encountered in <lambda> (vectorized)
  outputs = ufunc(*args, out=...)


...

/home/pbellec/git/cneuromod.all.qa_figures/.venv/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:2599: RuntimeWarning: invalid value encountered in <lambda> (vectorized)
  outputs = ufunc(*args, out=...)


/home/pbellec/git/cneuromod.all.qa_figures/.venv/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:2599: RuntimeWarning: invalid value encountered in <lambda> (vectorized)
  outputs = ufunc(*args, out=...)


/home/pbellec/git/cneuromod.all.qa_figures/.venv/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:2599: RuntimeWarning: invalid value encountered in <lambda> (vectorized)
  outputs = ufunc(*args, out=...)


/home/pbellec/git/cneuromod.all.qa_figures/.venv/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:2599: RuntimeWarning: invalid value encountered in <lambda> (vectorized)
  outputs = ufunc(*args, out=...)


/home/pbellec/git/cneuromod.all.qa_figures/.venv/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:2599: RuntimeWarning: invalid value encountered in <lambda> (vectorized)
  outputs = ufunc(*args, out=...)


In [5]:
def plot_tsnr(stat_map, title, out_path, display_mode, cut_coords, short_title=None):
    """Montage of one tSNR map (path or in-memory image) on the MNI template."""
    figure, annotation_size, title_size = montage_figure(out_path)
    title = montage_title(title, short_title, figure is not None)
    display = plotting.plot_stat_map(
        stat_map, display_mode=display_mode, cut_coords=cut_coords,
        cmap=CMAP, vmin=TSNR_VMIN, vmax=TSNR_VMAX, colorbar=True, black_bg=True,
        symmetric_cbar=False, figure=figure, annotate=False, title=None,
    )
    display.annotate(size=annotation_size)
    display.title(title, size=title_size)
    size_colorbar(display, annotation_size)
    display.savefig(str(out_path), dpi=figure_style.PAGE_DPI)
    display.close()


# One montage per dataset average per view (computed in memory, nothing written to disk).
for dataset, image in dataset_images.items():
    for view_suffix, display_mode, cut_coords in VIEWS:
        plot_tsnr(
            image, f"{dataset} — average tSNR",
            FIG_DIR / f"{dataset}_avgtsnr{view_suffix}.png", display_mode, cut_coords,
        )

# Grand average montage, pooling every subject across every dataset.
if grand_average_image is not None:
    for view_suffix, display_mode, cut_coords in VIEWS:
        plot_tsnr(grand_average_image, "all datasets — average tSNR",
                  FIG_DIR / f"all_datasets_avgtsnr{view_suffix}.png", display_mode,
                  cut_coords, short_title="average tSNR")


/home/pbellec/git/cneuromod.all.qa_figures/.venv/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:2599: RuntimeWarning: invalid value encountered in <lambda> (vectorized)
  outputs = ufunc(*args, out=...)


In [6]:
# One montage per subject, per dataset, per view — read straight from source_data,
# sliced at the same fixed cut coordinates as every other panel for comparability.
for dataset in datasets:
    for path in subject_maps(dataset):
        subject = path.name.split("_", 1)[0]  # e.g. sub-01
        for view_suffix, display_mode, cut_coords in VIEWS:
            plot_tsnr(str(path), f"{dataset} — {subject} tSNR",
                      FIG_DIR / f"{dataset}_{subject}_avgtsnr{view_suffix}.png",
                      display_mode, cut_coords)


/home/pbellec/git/cneuromod.all.qa_figures/.venv/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:2599: RuntimeWarning: invalid value encountered in <lambda> (vectorized)
  outputs = ufunc(*args, out=...)


/home/pbellec/git/cneuromod.all.qa_figures/.venv/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:2599: RuntimeWarning: invalid value encountered in <lambda> (vectorized)
  outputs = ufunc(*args, out=...)


/home/pbellec/git/cneuromod.all.qa_figures/.venv/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:2599: RuntimeWarning: invalid value encountered in <lambda> (vectorized)
  outputs = ufunc(*args, out=...)


/home/pbellec/git/cneuromod.all.qa_figures/.venv/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:2599: RuntimeWarning: invalid value encountered in <lambda> (vectorized)
  outputs = ufunc(*args, out=...)


/home/pbellec/git/cneuromod.all.qa_figures/.venv/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:2599: RuntimeWarning: invalid value encountered in <lambda> (vectorized)
  outputs = ufunc(*args, out=...)


/home/pbellec/git/cneuromod.all.qa_figures/.venv/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:2599: RuntimeWarning: invalid value encountered in <lambda> (vectorized)
  outputs = ufunc(*args, out=...)


/home/pbellec/git/cneuromod.all.qa_figures/.venv/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:2599: RuntimeWarning: invalid value encountered in <lambda> (vectorized)
  outputs = ufunc(*args, out=...)


/home/pbellec/git/cneuromod.all.qa_figures/.venv/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:2599: RuntimeWarning: invalid value encountered in <lambda> (vectorized)
  outputs = ufunc(*args, out=...)


/home/pbellec/git/cneuromod.all.qa_figures/.venv/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:2599: RuntimeWarning: invalid value encountered in <lambda> (vectorized)
  outputs = ufunc(*args, out=...)
